In [ ]:


# 下载 hsc的数据集
import os
from urllib.request import urlretrieve


def get_hsc_data(ra,dec,save_path):

   # hsc 的数据集   
   jpg_url = f"https://www.legacysurvey.org/viewer/cutout.jpg?ra={ra}&dec={dec}&layer=hsc-dr3&pixscale=0.168&size=512&bands=grizy"
   fits_url = f"https://www.legacysurvey.org/viewer/cutout.fits?ra={ra}&dec={dec}&layer=hsc-dr3&pixscale=0.168&size=512"

   save_path_fits = os.path.join(save_path,f"cutout_{ra}_{dec}.fits")
   save_path_jpg = os.path.join(save_path,f"cutout_{ra}_{dec}.jpg")

   if not os.path.exists(save_path):
      os.makedirs(save_path)

   if os.path.exists(save_path_fits) and os.path.exists(save_path_jpg):
      print(f"{save_path_fits} and {save_path_jpg} already exists")
   else:
      urlretrieve(jpg_url,save_path_jpg)
      urlretrieve(fits_url,save_path_fits)
      print(f"Downloaded {save_path_fits} and {save_path_jpg}")

   return save_path_fits,save_path_jpg

def get_sdss_data(ra,dec,save_path):

   # hsc 的数据集   
   jpg_url = f"https://www.legacysurvey.org/viewer/cutout.jpg?ra={ra}&dec={dec}&layer=sdss&pixscale=0.396&size=512"
   fits_url = f"https://www.legacysurvey.org/viewer/cutout.fits?ra={ra}&dec={dec}&layer=sdss&pixscale=0.396&size=512"

   save_path_fits = os.path.join(save_path,f"cutout_{ra}_{dec}.fits")
   save_path_jpg = os.path.join(save_path,f"cutout_{ra}_{dec}.jpg")

   if not os.path.exists(save_path):
      os.makedirs(save_path)

   if os.path.exists(save_path_fits) and os.path.exists(save_path_jpg):
      print(f"{save_path_fits} and {save_path_jpg} already exists")
   else:
      urlretrieve(jpg_url,save_path_jpg)
      urlretrieve(fits_url,save_path_fits)
      print(f"Downloaded {save_path_fits} and {save_path_jpg}")

   return save_path_fits,save_path_jpg

if __name__ == "__main__":
   ra = 172.0551
   dec = 4.3165
   save_path = "./hsc"
   get_hsc_data(ra,dec,save_path)

In [ ]:
# 可视化 fits 文件（zscale）
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval

def visualize_fits_zscale(fits_path):
    with fits.open(fits_path) as hdul:
        image_data = hdul[0].data.astype(np.float32)
        print(hdul[0].header)

    # zscale 归一化
    interval = ZScaleInterval()
    plt.figure(figsize=(12, 4))
    for i in range(3):
        band = image_data[i] if image_data.ndim == 3 else image_data
        vmin, vmax = interval.get_limits(band)
        plt.subplot(1, 3, i+1)
        plt.imshow(band, cmap='gray', vmin=vmin, vmax=vmax)
        plt.title(f'Band {i+1}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# grz 三波段
visualize_fits_zscale('D:\Program\月球大模型-地球化学所\标签修改\hsc\cutout_172.0551_4.3165.fits')

In [ ]:
from astroquery.mast import MastMissions
from astropy.coordinates import SkyCoord

# Create MastMissions object and assign mission to 'jwst'
missions = MastMissions(mission='jwst')

print(f'Mission: {missions.mission}')
print(f'Service: {missions.service}')

# Get available columns for JWST mission
columns = missions.get_column_list()
columns[:10]

# Create coordinate object
coords = SkyCoord(210.80227, 54.34895, unit=('deg'))

# Query for results within 10 arcminutes of coords
results = missions.query_region(coords, radius=10)

# Display results
print(f'Total number of results: {len(results)}')
results[:5]

In [ ]:

import tarfile
from urllib.request import urlretrieve

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from astropy.io import fits

from astropy.wcs import WCS
from astroquery.mast.missions import MastMissions
from bs4 import BeautifulSoup

# HST 数据： https://mast.stsci.edu/search/ui/#/hst
#       下载的是ACS的数据


def Unzip_file(path, suffix):
    directory = path
    suffix = suffix
    for filename in os.listdir(directory):
        if filename.endswith(suffix):
            filepath = os.path.join(directory, filename)
            with tarfile.open(filepath, 'r:bz2') as tar:
                tar.extractall(directory)
            os.remove(filepath)


def save_fit(img, path):
    if os.path.exists(path):
        os.remove(path)
    grey = fits.PrimaryHDU(img)
    greyHDU = fits.HDUList([grey])
    greyHDU.writeto(path)


def read_fits(path):
    hdu = fits.open(path, ignore_missing_simple=True)
    img = hdu[0].data
    img = np.array(img, dtype=np.float32)
    hdu.close()
    return img


def pltimage(data):
    plt.imshow(data)
    plt.show()


def get_jpg(url, file_name):
    r = requests.get(url)
    with open(file_name, "wb") as code:
        code.write(r.content)

class DataRadecHst(object):

    def __init__(self, ra, dec, size, savepath, band="r"):
        """
        reference:
        :param ra: RA of the target in degrees
        :param dec: Dec of the target in degrees
        :param band: desired filter (g, ,i, r, u, or z)
        :return:
        """
        self.ra, self.dec, self.band, self.size, self.savepath = ra, dec, band, size, savepath

        self.fitsfilename = self.savepath + "/fits/"+ f'cutout_{self.ra}_{self.dec}.fits'
        self.jpgfilename = self.savepath + "/jpg/" + f'cutout_{self.ra}_{self.dec}.jpg'

    def downloader(self):
        if self.get_url_web():
            if self.imshow():
                return True
        else:
            return False


    def get_url_web(self):

        missions = MastMissions(mission='hst')

        target = '{} {}'.format(self.ra, self.dec)
        radius = '1'
        select_cols = ['ang_sep', 'sci_aper_1234', 'sci_central_wavelength', 'sci_data_set_name', 'sci_dec',
                       'sci_actual_duration', 'sci_spec_1234', 'sci_hlsp', 'sci_instrume', 'sci_pi_last_name',
                       'sci_preview_name', 'sci_pep_id', 'sci_ra', 'sci_refnum', 'sci_release_date', 'scp_scan_type',
                       'sci_start_time', 'sci_stop_time', 'sci_targname']

        results = missions.query_object(
            target,
            radius=radius,
            select_cols=select_cols,
            sci_spec_1234='F814W,F814W;*,*;F814W,*;F814W;*',
            sci_obs_type='image',
            sci_aec='S',
            sci_instrume='acs,foc,nicmos,wfpc,wfpc2')

        if len(results) > 0:
            base_url = "https://mast.stsci.edu/search/hst/api/v0.1/retrieve_product?"
            fits_urls = [base_url + "product_name={}%2F{}_drc.fits".format(per, per.lower()) for per in
                         results["sci_data_set_name"]]
            fit_names = self.get_HST_fits(fits_urls)
            return True
        else:
            return False

    def get_HST_fits(self, download_fits_urls):

        file_names = []
        for d in download_fits_urls:
            print(d)
            r = requests.get(d)  # 根据文件的大小,这一步为主要耗时步骤
            print(r.status_code)
            if r.status_code == 200:
                file_name = d[-20:]
                file_name = os.path.join(self.savepath, file_name)
                with open(file_name, "wb") as code:
                    code.write(r.content)
                file_names.append(file_name)
            else:
                pass
            file_name = d[-20:]
            file_name = os.path.join(self.savepath, file_name)
            file_names.append(file_name)

        return file_names

    def imshow(self):

        data_path = self.file_names[0]
        hdulist = fits.open(data_path, ignore_missing_simple=True)
        wcs = WCS(hdulist[1].header)
        image = hdulist[1].data
        x, y = wcs.all_world2pix(self.ra, self.dec, 1)
        hdulist.close()

        if y > self.size and x > self.size:
            image1 = image[int(y - self.size): int(y + self.size), int(x - self.size):int(x + self.size)]
            save_fit(image1, self.fitsfilename)
            return True
        else:
            image1 = image[int(y - self.size): int(y + self.size), int(x - self.size):int(x + self.size)]
            save_fit(image1, self.fitsfilename)
            return False

    def fitsimshow(self):
        data = read_fits(self.fitsfilename)
        return data


In [1]:
from astroquery.mast import Observations
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np
from astropy.table import vstack, unique
import time
! pip install tqdm
import tqdm

# 中心坐标
center_ra = 270.1  # deg
center_dec = 66.2  # deg
center = SkyCoord(center_ra, center_dec, unit='deg')

# 区域参数
total_radius_deg = 5  # 覆盖 ±2.5°，总宽5°
search_radius = 1 * u.arcmin  # 每次查询半径
step_size_deg = 0.01  # 步长（度）

# 考虑球面效应：RA方向步长需除以 cos(dec)
cos_dec = np.cos(np.radians(center_dec))
ra_step = step_size_deg / cos_dec
dec_step = step_size_deg

# 生成网格
ra_min = center_ra - total_radius_deg
ra_max = center_ra + total_radius_deg
dec_min = center_dec - total_radius_deg
dec_max = center_dec + total_radius_deg

ra_points = np.arange(ra_min, ra_max + ra_step/2, ra_step)
dec_points = np.arange(dec_min, dec_max + dec_step/2, dec_step)

print(f"RA points: {len(ra_points)}, Dec points: {len(dec_points)}")
print(f"Total tiles: {len(ra_points) * len(dec_points)}")

# 存储所有结果
all_obs = []

# 筛选条件函数
def filter_hst_acs(obs_table):
    if len(obs_table) == 0:
        return obs_table
    mask = (
        (obs_table["instrument_name"] == "ACS/WFC") &
        (obs_table["dataproduct_type"] == "image") &
        (obs_table["obs_collection"] == "HST") &
        (obs_table["intentType"] == "science") &
        (obs_table["calib_level"] == 3)
    )
    return obs_table[mask]

# 逐个 tile 查询
for i, dec in tqdm.tqdm(enumerate(dec_points), desc="Processing tiles"):
    for j, ra in tqdm.tqdm(enumerate(ra_points), desc="Processing RA points"):
        coord = SkyCoord(ra % 360, dec, unit='deg')
        try:
            obs = Observations.query_region(coord, radius=search_radius)
            hsc = filter_hst_acs(obs)
            if len(hsc) > 0:
                all_obs.append(hsc)
                print(f"Tile ({i},{j}): RA={ra:.3f}, Dec={dec:.3f} → {len(hsc)} results")
            time.sleep(0.5)
        except Exception as e:
            print(f"Error at RA={ra}, Dec={dec}: {e}")
            continue

if all_obs:
    final_table = vstack(all_obs)
    final_table_unique = unique(final_table, keys='obs_id')
    print(f"\nTotal unique HST/ACS/WFC images found: {len(final_table_unique)}")
else:
    print("No observations found.")
    final_table_unique = None

# 可选：保存结果
# if final_table_unique:
#     final_table_unique.write('hst_acs_5deg_region.ecsv', format='ascii.ecsv', overwrite=True)


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


RA points: 405, Dec points: 1001
Total tiles: 405405


Processing RA points: 16it [09:04, 34.03s/it]
Processing tiles: 0it [09:04, ?it/s]


KeyboardInterrupt: 

In [1]:
all_obs

NameError: name 'all_obs' is not defined

In [ ]:
from astroquery.mast import Observations
from astropy.coordinates import SkyCoord
import astropy.units as u

# EDF - N
coord = SkyCoord(270.1, 66.2, unit='deg')
obs = Observations.query_region(
    coord,
    radius=5 * u.arcmin,
)
hsc = obs[(obs["instrument_name"] == "ACS/WFC") & (obs["dataproduct_type"] == "image") & (obs["obs_collection"] == "HST") & (obs["intentType"] == "science")& (obs["calib_level"] == 3)]

In [2]:
set(obs["obs_collection"])


{np.str_('GALEX'),
 np.str_('HLSP'),
 np.str_('HST'),
 np.str_('PS1'),
 np.str_('TESS')}

In [3]:
obs[(obs["dataproduct_type"] == "image") & (obs["obs_collection"] == "HST") ]

intentType,obs_collection,provenance_name,instrument_name,project,filters,wave_region,target_name,target_classification,obs_id,s_ra,s_dec,dataproduct_type,proposal_pi,calib_level,t_min,t_max,t_exptime,wavelength_region,em_min,em_max,obs_title,t_obs_release,proposal_id,proposal_type,sequence_number,s_region,jpegURL,dataURL,dataRights,mtFlag,srcDen,obsid,wave_min,wave_max,distance
str11,str5,str7,str10,str4,str11,str16,str29,str1,str65,float64,float64,str10,str20,int64,float64,float64,float64,str16,float64,float64,str64,float64,str23,str7,int64,str347,str156,str150,str6,bool,float64,str9,float64,float64,float64
calibration,HST,CALACS,ACS/WFC,HST,F502N;F660N,Optical,BIAS,--,jbvvap010,201.7030032798,-47.48460889257,image,"Golimowski, David A.",3,56004.00162581018,56004.01620914352,0.0,Optical,499.40000000000003,661.0,CCD Daily Monitor {Part 2},56004.1787847,12782,CAL/ACS,--,POLYGON -89.892473999999993 66.255763 -89.889596999999981 66.270282 -89.923842999999977 66.270359 -89.92670099999998 66.25584 -89.892473999999993 66.255763 -89.892473999999993 66.255763 POLYGON -158.396347 -47.571202 -158.38530200000002 -47.582796 -158.366329 -47.575843 -158.37737700000002 -47.564252 -158.396347 -47.571202 -158.396347 -47.571202,--,--,PUBLIC,False,nan,26123423,499.40000000000003,661.0,200.8067706634409
